# Example 9: 2-D tracking and post-hoc animation
This notebook walks through the same 2×2 grid example as Example 6
(`06_2D_quickStart`), but this time using the 2-D tracking function
`run_VDERM_2d_with_tracking` to export intermediate states, and then the
faster `animate_map_posthoc` function to approximate the same animation
without re-interpolating at every tracked iteration.

This is the 2-D analog of Example 4 (`04_tracking`).

In [ ]:
import diffusion_cartogram as vd
import numpy as np

### Create four rectangular regions

We define four 1×1 squares arranged in a 2×2 pattern and sample points along
each perimeter — the same setup as Example 6.

```
┌──────┬──────┐
│  C   │  D   │  y ∈ [1, 2]
│  ρ=2 │ ρ=10 │
├──────┼──────┤
│  A   │  B   │  y ∈ [0, 1]
│  ρ=1 │  ρ=5 │
└──────┴──────┘
 x∈[0,1] x∈[1,2]
```

In [ ]:
def region_boundary(x0, x1, y0, y1, n=40):
    """Return n points sampled along the perimeter of a rectangle."""
    bottom = np.column_stack([np.linspace(x0, x1, n), np.full(n, y0)])
    top    = np.column_stack([np.linspace(x0, x1, n), np.full(n, y1)])
    left   = np.column_stack([np.full(n, x0), np.linspace(y0, y1, n)])
    right  = np.column_stack([np.full(n, x1), np.linspace(y0, y1, n)])
    return np.vstack([bottom, top, left, right])

A = region_boundary(0, 1, 0, 1)   # lower-left,  rho = 1
B = region_boundary(1, 2, 0, 1)   # lower-right, rho = 5
C = region_boundary(0, 1, 1, 2)   # upper-left,  rho = 2
D = region_boundary(1, 2, 1, 2)   # upper-right, rho = 10

map_points = np.vstack([A, B, C, D])
print(f"Total map boundary points: {len(map_points)}")

In [ ]:
vd.plot_map_2d(map_points, title='2×2 Grid — Initial Layout')

### Create the underlying VDERM grid and assign densities

Same as Example 6: `make_initial_grid_2d` sizes a grid around the point
cloud, and `quadrant_density` assigns a density based on which of the four
regions a grid node falls in.

In [ ]:
grid_params = vd.make_initial_grid_2d(map_points, max_points=4096)
vd.print_grid_info_2d(grid_params)

In [ ]:
def quadrant_density(x, y):
    """Return density based on which 1×1 region contains (x, y)."""
    if y >= 0 and y <= 2.0 and x >= 0 and x <= 2.0:
        if x >= 1.0 and x <= 2.0:
            return 10.0 if y >= 1.0 else 5.0   # D or B
        elif x >= 0:
            return 2.0  if y >= 1.0 else 1.0   # C or A
    else:
        return np.mean([1, 2, 5, 10])

grid = vd.VDERMGrid2D(
    shape=grid_params['shape'],
    h=grid_params['h'],
    min_bounds=grid_params['min_bounds'],
)
grid.set_density(quadrant_density)

fig = vd.plot_density_field_2d(grid, title='Initial Density Field (4 Regions)')

## Run VDERM with tracking

`run_VDERM_2d_with_tracking` is the 2-D analog of `run_VDERM_with_tracking`
used in Example 4. It periodically exports the grid state and/or the
deformed map points as the deformation progresses, which lets us build gifs
and density plots afterward. Each export requires interpolating the
displacement field onto the map points, so exporting more frequently (or for
more iterations) costs more time.

In [ ]:
final_grid = vd.run_VDERM_2d_with_tracking(
    grid, map_points, n_max=300, max_eps=0.01,
    export_grid=True, export_grid_frequency=10,
    export_map=True, export_map_frequency=10,
    base_folder='my_2d_deformation'
)

## Export plots and gifs

Just like the 3-D pipeline, we can animate the tracked grid and map states,
and plot how the density field equalizes over time.

In [ ]:
vd.animate_grid_deformation_2d('my_2d_deformation', output_file='grid_2d.gif', fps=10, subsample=None)
vd.animate_map_deformation_2d('my_2d_deformation', output_file='map_2d.gif', fps=10, subsample=None)
vd.plot_density_evolution_2d('my_2d_deformation', output_file='density_2d_plot.png')

## Post-hoc map animation (fast alternative)

The map animation above required `run_VDERM_2d_with_tracking` to interpolate
the displacement field onto the map points every 10 iterations. `animate_map_posthoc`
instead interpolates the map **only twice** — once at the start, once at the
final converged state — and fills in the requested number of intermediate
frames by easing point positions (and densities, since we pass
`initial_densities=quadrant_density`) between those two states on a
`1 - exp(-t/tau)` timescale, so motion appears fast at first and decelerates,
approximating the real VDERM dynamics without needing to know them exactly.

It's much cheaper, but the intermediate frames are an approximation, not a
physically accurate reconstruction — the function prints a notice to that
effect every time it runs. Because it writes frames using the same
`map_iteration_NNNN.csv` / `map_final_iteration_NNNN.csv` convention as
`run_VDERM_2d_with_tracking`, the output folder can be passed straight into
`animate_map_deformation_2d` exactly as before.

In [ ]:
vd.animate_map_posthoc(
    final_grid, map_points, n_frames=25,
    output_folder='my_2d_deformation_posthoc',
    initial_densities=quadrant_density
)
vd.animate_map_deformation_2d('my_2d_deformation_posthoc', output_file='map_2d_posthoc.gif',
                              fps=10, subsample=None)

### Comparing the two animations

`map_2d.gif` (from tracking) and `map_2d_posthoc.gif` (from the post-hoc
approximation) should look broadly similar for this simple example, since
the deformation here is close to monotonic. For deformations with more
complex, non-monotonic intermediate motion, the tracked animation remains
the accurate reference — reach for `animate_map_posthoc` when you want a
quick preview or a display-quality animation without paying for repeated
interpolation.